In [ ]:
import zarr
import dask.array as da
import numpy as np
import pandas as pd
import os
from torch.utils.data import Dataset
import torch
import pytorch_lightning as pl
from pathlib import Path
import json

import matplotlib.pyplot as plt

# Dataloader and Datamodule Classes.

These classes have already been included in the data_loader.py source file. They're included here for debugging convenience.

In [ ]:
class ZarrIrradianceDataset(Dataset):

    def __init__(self, aligndata, aia_zarr_path, eve_zarr_path, wavelengths, ions, freq, months, transformations=None):
        """
        aia_zarr_path --> path: path to zarr aia data
        eve_zarr_path --> path: path to zarr eve data
        wavelengths   --> list: list of channels for aia
        ions          --> list: list of ions for eve
        freq          --> str: cadence used for rounding time series
        transformation: to be applied to aia in theory, but can stay None here
        """
        
        self.aligndata = aligndata
        self.aia_zarr_path = aia_zarr_path
        self.eve_zarr_path = eve_zarr_path
        self.wavelengths = wavelengths
        self.ions = ions
        self.cadence = freq
        self.months = months
        self.transformations = transformations

        # get data from path
        self.aia_data = zarr.group(zarr.DirectoryStore(self.aia_zarr_path))
        self.eve_data = zarr.group(zarr.DirectoryStore(self.eve_zarr_path))

        self.aligndata = self.aligndata.loc[self.aligndata.index.month.isin(self.months),:]
        

    def __len__(self):
        return self.aligndata.shape[0]
    

    def __getitem__(self, idx):
        index_row = self.aligndata.iloc[idx,:]
        aia_image_list = []

        for wavelength in self.wavelengths:
            # select data from zarr
            aia_channel = self.aia_data[index_row[f'year_{wavelength}']][wavelength]
            
            # convert data to dask
            aia_image_list.append(torch.tensor(np.array(da.from_array(aia_channel)[index_row[f'index_{wavelength}'],:,:])))

        euv_images = torch.stack(aia_image_list)
    
        if self.transformations is not None:
            # transform as RGB + y to use transformations
            euv_images = euv_images.transpose() # this is probably needed for the dimension of the file they are passing
            # transformed = self.transformations(image=euv_images[..., :3], y=euv_images[..., 3:])
            # euv_images = torch.cat([transformed['image'], transformed['y']], dim=0)
            transformed = self.transformations(image=euv_images) # this applies transformations (here it just "toTensor", but 
                                                                # it could be like rotating, change colors etc..)
            euv_images = transformed['image']
        
        eve_ion_list = []
        for ion in self.ions:
            eve_ion_list.append(torch.tensor(np.array(da.from_array(self.eve_data['MEGS-A'][ion])[index_row['index_eve']])))
        
        eve_data = torch.stack(eve_ion_list)

        return euv_images, eve_data


    def get_aia_image(self, idx):

        aia_image_list = {}
        for wavelength in self.wavelengths:
            idx_mapped = self.aligndata[f'index_{wavelength}'][idx]
            year = idx_mapped.index.year[0]
            aia_image_list[wavelength] = self.aia_data[year][wavelength][idx_mapped, :, :]
            aia_image_list[wavelength] -= self.normalizations["AIA"][wavelength][year]["mean"]
            # aia_image_list[wavelength] /= self.normalizations["AIA"][wavelength]["2010"]["max"]
        return 
    

    def get_eve(self, idx):
        eve_ion_dict = {}
        for ion in self.ions:
            idx_mapped = self.aligndata['index_eve'][idx]
            eve_ion_dict[ion] = self.eve_data['MEGS-A'][ion][idx_mapped]
            eve_ion_dict[ion] -= self.normalizations["EVE"][ion]["mean"]
            eve_ion_dict[ion] /= self.normalizations["EVE"][ion]["std"]
        return eve_ion_dict


class ZarrIrradianceDataModule(pl.LightningDataModule):
    def __init__(self, aia_path, eve_path, wavelengths, ions, frequency, batch_size: int = 32, num_workers=None,
                 train_transforms=None, val_transforms=None, val_months=[10,1], test_months=[11,12], holdout_months=None):
        """ Loads paired data samples of AIA EUV images and EVE irradiance measures.

        Note: Input data needs to be paired.
        Parameters
        ----------
        aia_path: path to aia zarr file
        eve_path: path to the EVE zarr data file
        batch_size: batch size (default is 32)
        num_workers: number of workers (needed for the training)
        train_transforms: transformations to be applied on the training set
        val_transforms: transformations to be applied on the validation set
        val_months: 
        """

        super().__init__()
        self.num_workers = num_workers if num_workers is not None else os.cpu_count() // 2
        self.aia_path = aia_path
        self.eve_path = eve_path
        self.batch_size = batch_size
        self.train_transforms = train_transforms
        self.val_transforms = val_transforms
        self.wavelengths = wavelengths
        self.wavelengths.sort()
        self.ions = ions
        self.ions.sort()
        self.cadence = frequency
        self.val_months = val_months
        self.test_months = test_months
        self.holdout_months = holdout_months

        # Chunk size depends on ram available. 700 is a good number for 16GB of ram and 30min cadence.
        # If you have more ram, at same cadence, you can increase this number to speed up the normalization calculation.
        self.zarr_chunk_size = 700 

        # Cache filenames
        wavelength_id = "_".join(self.wavelengths)
        ions_id = "_".join(ions).replace(" ", "_")
        self.cache_id = f"{wavelength_id}_{ions_id}_{self.cadence}"
        if "small" in self.aia_path: 
            self.cache_id += "_small"

        self.index_cache_filename = f"aligndata_{self.cache_id}.csv"
        self.normalizations_cache_filename = f"normalizations_{self.cache_id}.json"

        self.aia_data = zarr.group(zarr.DirectoryStore(self.aia_path))
        self.eve_data = zarr.group(zarr.DirectoryStore(self.eve_path))

        # Temporal alignment of aia and eve data
        self.aligndata = self.__aligntime__()

        # Calculate normalization of the data
        self.normalizations = self.__calc_normalizations__()


    def __aligntime__(self):
        """
        This function extracts the common indexes across aia and eve datasets, considering potential missing values.
        """

        if Path(self.index_cache_filename).exists():
            aligndata = pd.read_csv(self.index_cache_filename)
            aligndata["Time"] = pd.to_datetime(aligndata["Time"])
            aligndata.set_index("Time", inplace=True)
            return aligndata

        # select data from zarr
        for i,wavelength in enumerate(self.wavelengths):
            for j,key in enumerate(self.aia_data.keys()):
                aia_channel = self.aia_data[key][wavelength]

                # get observation time
                t_obs_aia_channel = aia_channel.attrs['T_OBS'] 
                if j == 0:
                    # transform to DataFrame
                    # AIA
                    df_t_aia = pd.DataFrame({'Time': pd.to_datetime(t_obs_aia_channel,format='mixed'), f'index_{wavelength}': np.arange(0,len(t_obs_aia_channel))})
                    df_t_aia[f'year_{wavelength}'] = key

                else:
                    df_tmp_aia =pd.DataFrame({'Time': pd.to_datetime(t_obs_aia_channel, format='mixed'), f'index_{wavelength}': np.arange(0,len(t_obs_aia_channel))})
                    df_tmp_aia[f'year_{wavelength}'] = key
                    df_t_aia = pd.concat([df_t_aia, df_tmp_aia], ignore_index = True)
            
            # enforcing same datetime format
            transform_datetime = lambda x: pd.to_datetime(x, format='mixed').strftime('%Y-%m-%d %H:%M:%S')
            df_t_aia['Time'] = df_t_aia['Time'].apply(transform_datetime)
            df_t_aia['Time'] = pd.to_datetime(df_t_aia['Time']).dt.tz_localize(None) # this is needed for timezone-naive type
            df_t_aia['Time'] = df_t_aia['Time'].dt.round(self.cadence)
            df_t_obs_aia = df_t_aia.drop_duplicates(subset='Time', keep='first') # removing potential duplicates derived by rounding
            df_t_obs_aia.set_index('Time', inplace = True)
            
            if i == 0:
                join_series = df_t_obs_aia
            else:
                join_series = join_series.join(df_t_obs_aia, how='inner')

        # EVE
        t_obs_eve_channel = np.array(da.from_array(self.eve_data['MEGS-A']['Time'])).tolist()
        df_t_eve = pd.DataFrame({'Time': pd.to_datetime(t_obs_eve_channel), 'index_eve': np.arange(0,len(t_obs_eve_channel))})
        df_t_eve['Time'] = pd.to_datetime(df_t_eve['Time'])  
        df_t_eve['Time'] = df_t_eve['Time'].dt.round(self.cadence)
        df_t_obs_eve = df_t_eve.drop_duplicates(subset='Time', keep='first')
        df_t_obs_eve.set_index('Time', inplace = True)
        join_series = join_series.join(df_t_obs_eve, how='inner')

        # remove missing eve data (missing values are labeled with negative values)
        for ion in self.ions:
            ion_data = np.array(da.from_array(self.eve_data['MEGS-A'][ion]))
            join_series = join_series.loc[ion_data[join_series['index_eve']] > 0,:]

        join_series.to_csv(self.index_cache_filename)
        return join_series


    def __calc_normalizations__(self):

        if Path(self.normalizations_cache_filename).exists():
            with open(self.normalizations_cache_filename, "r") as json_file:
                return json.load(json_file)

        normalizations = {}

        # EVE Normalization
        normalizations["EVE"] = {}
        for ion in self.ions:
            # Note that selecting on idx self.aligndata['index_eve'] removes negative values from EVE data.
            channel_data = self.eve_data['MEGS-A'][ion][self.aligndata['index_eve']][:]
            normalizations["EVE"][ion] = {}
            normalizations["EVE"][ion]["count"] = channel_data.shape[0]
            normalizations["EVE"][ion]["sum"] = channel_data.sum()
            normalizations["EVE"][ion]["mean"] = channel_data.mean()
            normalizations["EVE"][ion]["std"] = channel_data.std()
            normalizations["EVE"][ion]["min"] = channel_data.min()
            normalizations["EVE"][ion]["max"] = channel_data.max()
    
        # AIA Normalization
        normalizations["AIA"] = {}
        for wavelength in self.wavelengths:
            normalizations["AIA"][wavelength] = {}
            for year in self.aia_data.keys():
                normalizations["AIA"][wavelength] = {}
                normalizations["AIA"][wavelength][year] = {}
                
                idx_channel = self.aligndata[f'index_{wavelength}']
                idx_channel = idx_channel.loc[idx_channel.index.year == int(year)]
                wavelength_data = self.aia_data[year][wavelength][idx_channel]

                normalizations["AIA"][wavelength][year]["sum"] = 0.
                normalizations["AIA"][wavelength][year]["count"] = 0
                normalizations["AIA"][wavelength][year]["min"] = float("inf")
                normalizations["AIA"][wavelength][year]["max"] = float("-inf")

                # Because AIA data is too big to fit in memory, we need to calculate the normalization in chunks.
                num_chunks = wavelength_data.shape[0] // self.zarr_chunk_size + 1

                for chunk_num in range(num_chunks):
                    idx_left = chunk_num * self.zarr_chunk_size
                    idx_right = min((chunk_num+1) * self.zarr_chunk_size, wavelength_data.shape[0])

                    chunk = wavelength_data[idx_left:idx_right, :, :].flatten()

                    normalizations["AIA"][wavelength][year]["count"] += chunk.shape[0]
                    normalizations["AIA"][wavelength][year]["sum"] += chunk.sum()
                    normalizations["AIA"][wavelength][year]["min"] = min(normalizations["AIA"][wavelength][year]["min"], chunk.min())
                    normalizations["AIA"][wavelength][year]["max"] = max(normalizations["AIA"][wavelength][year]["max"], chunk.max())


        with open(self.normalizations_cache_filename, "w") as json_file:
            save_json = str(normalizations)
            save_json = save_json.replace("'", '"')
            json_file.write(save_json)

        return normalizations


    def setup(self):
        self.train_months = [i for i in range(1,13) if i not in self.test_months + self.val_months]

        self.train_ds = ZarrIrradianceDataset(self.aligndata, self.aia_path, self.eve_path, 
                                              self.wavelengths, self.ions, self.cadence, self.train_months, self.train_transforms)
        
        self.test_ds = ZarrIrradianceDataset(self.aligndata, self.aia_path, self.eve_path, 
                                              self.wavelengths, self.ions, self.cadence, self.test_months)
        
        self.valid_ds = ZarrIrradianceDataset(self.aligndata, self.aia_path, self.eve_path, 
                                              self.wavelengths, self.ions, self.cadence, self.val_months, self.val_transforms)


data_loader = ZarrIrradianceDataModule(
    aia_path='/mnt/sdomlv2_small/sdomlv2_small.zarr', 
    eve_path='/mnt/sdomlv2_small/sdomlv2_eve.zarr',
    wavelengths=['171A','304A','193A'],
    ions=['Fe IX','Fe VIII'],
    frequency='30min',
    batch_size=32, 
    num_workers=None,
    train_transforms=None, 
    val_transforms=None, 
    val_months=[10,1], 
    test_months=[11,12], 
    holdout_months=None
)

data_loader.setup()

In [ ]:
idx = 0
idx_mapped = data_loader.aligndata['index_171A'][idx]
img_dict = {}
data_loader.wavelengths
img_dict["171A"] = data_loader.aia_data["2010"]["171A"][idx_mapped, :, :]
...

# create the 9 channel image:
return_img = torch.stack([img for img in img_dict.values()])

## Scratch

# Old _getitem_

def __getitem__(self, idx):
    index_row = self.aligndata.iloc[idx,:]
    aia_image_list = []

    for wavelength in self.wavelengths:
        # select data from zarr
        aia_channel = self.aia_data[index_row[f'year_{wavelength}']][wavelength]
        
        # convert data to dask
        aia_image_list.append(torch.tensor(np.array(da.from_array(aia_channel)[index_row[f'index_{wavelength}'],:,:])))

    euv_images = torch.stack(aia_image_list)

    if self.transformations is not None:
        # transform as RGB + y to use transformations
        euv_images = euv_images.transpose() # this is probably needed for the dimension of the file they are passing
        # transformed = self.transformations(image=euv_images[..., :3], y=euv_images[..., 3:])
        # euv_images = torch.cat([transformed['image'], transformed['y']], dim=0)
        transformed = self.transformations(image=euv_images) # this applies transformations (here it just "toTensor", but 
                                                            # it could be like rotating, change colors etc..)
        euv_images = transformed['image']
    
    eve_ion_list = []
    for ion in self.ions:
        eve_ion_list.append(torch.tensor(np.array(da.from_array(self.eve_data['MEGS-A'][ion])[index_row['index_eve']])))
    
    eve_data = torch.stack(eve_ion_list)

    return euv_images, eve_data